In [1]:
import xml.etree.ElementTree as ET
import pandas as pd
from pathlib import Path

def strip_ns(tag):
    return tag.split("}")[-1] if "}" in tag else tag

def parse_tcga_clinical_xml(filepath):
    tree = ET.parse(filepath)
    root = tree.getroot()
    record = {}

    def recurse(elem):
        children = list(elem)
        if not children:
            text = (elem.text or "").strip()
            tag = strip_ns(elem.tag)
            if text and (tag not in record or not record[tag]):
                record[tag] = text
        else:
            for c in children:
                recurse(c)

    recurse(root)
    return record

def parse_tcga_folder(folder, histology_label):
    records = []
    xml_files = list(Path(folder).rglob("nationwidechildrens.org_clinical.*.xml"))
    print(f"{folder}: found {len(xml_files)} clinical XML files")
    for f in xml_files:
        try:
            rec = parse_tcga_clinical_xml(f)
            rec["HISTOLOGY"] = histology_label
            rec["SOURCE_FILE"] = f.name
            records.append(rec)
        except Exception as e:
            print(f"Failed to parse {f.name}: {e}")
    return pd.DataFrame(records)

In [2]:
luad_df = parse_tcga_folder("datasets/TCGA-LUAD", "LUAD")
lusc_df = parse_tcga_folder("datasets/TCGA-LUSC", "LUSC")

tcga_nsclc = pd.concat([luad_df, lusc_df], ignore_index=True)
print(tcga_nsclc.shape)
print(tcga_nsclc["HISTOLOGY"].value_counts())
tcga_nsclc.to_csv("datasets/tcga_nsclc_clinical.csv", index=False)

datasets/TCGA-LUAD: found 522 clinical XML files
datasets/TCGA-LUSC: found 504 clinical XML files
(1026, 120)
HISTOLOGY
LUAD    522
LUSC    504
Name: count, dtype: int64


In [3]:
# See the full column list first
print(tcga_nsclc.columns.tolist())

['bcr', 'file_uuid', 'batch_number', 'project_code', 'disease_code', 'day_of_dcc_upload', 'month_of_dcc_upload', 'year_of_dcc_upload', 'withdrawn', 'tumor_tissue_site', 'histological_type', 'other_dx', 'gender', 'vital_status', 'days_to_birth', 'days_to_last_followup', 'race', 'bcr_patient_barcode', 'tissue_source_site', 'patient_id', 'bcr_patient_uuid', 'history_of_neoadjuvant_treatment', 'informed_consent_verified', 'icd_o_3_site', 'icd_o_3_histology', 'icd_10', 'tissue_prospective_collection_indicator', 'tissue_retrospective_collection_indicator', 'days_to_initial_pathologic_diagnosis', 'age_at_initial_pathologic_diagnosis', 'year_of_initial_pathologic_diagnosis', 'ethnicity', 'person_neoplasm_cancer_status', 'day_of_form_completion', 'month_of_form_completion', 'year_of_form_completion', 'system_version', 'pathologic_stage', 'pathologic_T', 'pathologic_N', 'pathologic_M', 'tobacco_smoking_history', 'anatomic_neoplasm_subdivision', 'diagnosis', 'kras_mutation_found', 'kras_gene_anal

In [4]:
# Then check completeness — this tells us which fields are actually usable
# vs. mostly empty/administrative noise
missingness = tcga_nsclc.isna().mean().sort_values()
print(missingness.head(40))   # most complete fields
print(missingness.tail(40))   # sparsest/least useful fields

bcr                                          0.000000
file_uuid                                    0.000000
batch_number                                 0.000000
project_code                                 0.000000
disease_code                                 0.000000
day_of_dcc_upload                            0.000000
month_of_dcc_upload                          0.000000
year_of_dcc_upload                           0.000000
withdrawn                                    0.000000
tumor_tissue_site                            0.000000
vital_status                                 0.000000
gender                                       0.000000
bcr_patient_uuid                             0.000000
patient_id                                   0.000000
tissue_source_site                           0.000000
bcr_patient_barcode                          0.000000
icd_o_3_site                                 0.000000
informed_consent_verified                    0.000000
icd_10                      

In [5]:
# Check missingness for specific fields not shown in the head/tail slices
cols_to_check = [
    "days_to_death", "days_to_last_followup", "days_to_last_known_alive",
    "number_pack_years_smoked", "year_of_tobacco_smoking_onset",
    "eastern_cancer_oncology_group", "karnofsky_performance_score",
    "race", "ethnicity", "drug_name", "therapy_type",
    "radiation_therapy", "new_tumor_event_after_initial_treatment"
]
print(tcga_nsclc[cols_to_check].isna().mean().sort_values())

radiation_therapy                          0.108187
new_tumor_event_after_initial_treatment    0.134503
race                                       0.175439
days_to_last_followup                      0.235867
number_pack_years_smoked                   0.236842
ethnicity                                  0.295322
year_of_tobacco_smoking_onset              0.415205
eastern_cancer_oncology_group              0.518519
days_to_death                              0.611111
therapy_type                               0.685185
drug_name                                  0.699805
karnofsky_performance_score                0.704678
days_to_last_known_alive                   0.925926
dtype: float64


In [6]:
final_cols = [
    # Identifiers
    "bcr_patient_barcode", "HISTOLOGY",
    # Demographics
    "gender", "race", "ethnicity", "age_at_initial_pathologic_diagnosis",
    # Staging
    "pathologic_stage", "pathologic_T", "pathologic_N", "pathologic_M",
    # Diagnosis
    "histological_type", "icd_o_3_histology", "tumor_tissue_site",
    # Smoking
    "tobacco_smoking_history", "number_pack_years_smoked",
    # Molecular
    "egfr_mutation_result", "kras_mutation_result", "eml4_alk_translocation_performed",
    # Survival
    "vital_status", "days_to_death", "days_to_last_followup",
    # Functional status (partial coverage — see note above)
    "eastern_cancer_oncology_group", "karnofsky_performance_score",
    # Treatment context
    "radiation_therapy", "new_tumor_event_after_initial_treatment",
]

tcga_clean = tcga_nsclc[final_cols].copy()

# Build a unified survival time column, same pattern as MSK-CHORD's OS_MONTHS/OS_STATUS
tcga_clean["days_to_event"] = tcga_clean["days_to_death"].fillna(tcga_clean["days_to_last_followup"])
tcga_clean["OS_STATUS"] = tcga_clean["vital_status"].map({"Dead": "1:DECEASED", "Alive": "0:LIVING"})

print(tcga_clean.shape)
print(tcga_clean["OS_STATUS"].value_counts(dropna=False))
print(tcga_clean["days_to_event"].isna().sum(), "patients with no usable survival time")

tcga_clean.to_csv("datasets/tcga_nsclc_clinical_clean.csv", index=False)

(1026, 27)
OS_STATUS
0:LIVING      738
1:DECEASED    288
Name: count, dtype: int64
15 patients with no usable survival time
